# 1. Business Question and Target Variable

## Primary Business Question
Can we accurately predict the SalePriceUSD of a property using its physical characteristics (size, age, condition), neighborhood context, and prevailing market conditions (mortgage rates) to       provide real estate agents with data-driven pricing benchmarks?
### Target Definition
  - **Target Variable**: SalePriceUSD
  - **Definition**: The final recorded transaction price of the property in U.S. Dollars.

### Why This Target Matters
Predicting the sale price is the core objective of real estate analytics. From a business perspective, an accurate prediction allows the agency to:

1) Establish competitive initial listing prices to minimize "time on market."

2) Identify properties that are currently underpriced or overpriced relative to their features.

3) Forecast potential revenue and agent commissions for better financial planning.

### What One Row Represents
In this dataset, a single row represents a unique property listing that has been sold, containing its specific attributes (e.g., SqFt, Beds, Baths), its location context (Neighborhood, DistanceToCityMiles), and the financial environment at the time of sale (MortgageRatePct).

### Columns to be Removed
To ensure the model generalizes well and adheres to best practices, the following columns will be dropped:
1) ListingID: This is a unique identifier/index for each row. It contains no predictive patterns and would lead to overfitting if included.
2) PostSaleAppraisalUSD: This column must be removed to prevent Data Leakage.

### Data Leakage Identification
- **Leakage Column**: PostSaleAppraisalUSD
- **Reason**: This value is determined after the sale has occurred. In a real-world predictive scenario, we would not know the post-sale appraisal value at the time we are trying to predict the listing price. Including it would artificially inflate the model's accuracy because the appraisal is directly informed by the final sale price.

In [5]:
import numpy as np
import pandas as pd
# Create plots/visualizations (e.g., histograms, scatter plots, residual plots)
import matplotlib.pyplot as plt

# Split data into train/test sets; optionally compute cross-validated scores
from sklearn.model_selection import train_test_split, cross_val_score  
# Apply different preprocessing steps to different column groups (numeric vs categorical)
from sklearn.compose import ColumnTransformer  
# Chain preprocessing + modeling steps into one reproducible workflow
from sklearn.pipeline import Pipeline  
# Encode categorical variables; standardize numeric features (mean 0, std 1)
from sklearn.preprocessing import OneHotEncoder, StandardScaler  
# Fill missing values using a chosen strategy (e.g., median, most_frequent)
from sklearn.impute import SimpleImputer
# Fit a Multiple Linear Regression model
from sklearn.linear_model import LinearRegression
# Evaluate regression predictions using MAE, MSE/RMSE, and R²
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 

In [6]:
path = "IST660_IA4_synthetic_real_estate.csv"

# Load the data
# keep_default_na=False ensures specific strings are not auto-converted, 
# allowing us to handle "blanks to NaN" manually as required by instructions.
df = pd.read_csv(path, keep_default_na=False)  

# Display basic structural information
print("Shape:", df.shape)
print("Shape (rows, cols):", df.shape)
print("\nColumns in dataset:", df.columns.tolist())
# Check data types and non-null counts
df.info()
display(df.head(5))
display(df.tail(5))

Shape: (900, 17)
Shape (rows, cols): (900, 17)

Columns in dataset: ['ListingID', 'Neighborhood', 'PropertyType', 'Condition', 'SqFt', 'LotSqFt', 'Beds', 'Baths', 'YearBuilt', 'DistanceToCityMiles', 'SchoolRating', 'CrimeIndex', 'MortgageRatePct', 'TotalRooms', 'OnlineEstimateUSD', 'SalePriceUSD', 'PostSaleAppraisalUSD']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ListingID             900 non-null    int64  
 1   Neighborhood          900 non-null    object 
 2   PropertyType          900 non-null    object 
 3   Condition             900 non-null    object 
 4   SqFt                  900 non-null    object 
 5   LotSqFt               900 non-null    object 
 6   Beds                  900 non-null    int64  
 7   Baths                 900 non-null    object 
 8   YearBuilt             900 non-null    int64  
 9   Distan

,ListingID,Neighborhood,PropertyType,Condition,SqFt,LotSqFt,Beds,Baths,YearBuilt,DistanceToCityMiles,SchoolRating,CrimeIndex,MortgageRatePct,TotalRooms,OnlineEstimateUSD,SalePriceUSD,PostSaleAppraisalUSD
0,500001,Rural,House,Good,2216,9938,3,2.5,1956,14.19,4.8,48.5,6.36,9.5,487927.0,490004.0,459168.0
1,500002,University,House,Good,3028,5609,4,2.0,2022,9.43,7.6,38.4,4.97,8.0,738356.0,786742.0,788502.0
2,500003,Rural,Townhome,Good,2035,5331,3,2.5,2008,8.75,6.0,33.5,7.35,7.5,477247.0,467777.0,483201.0
3,500004,Suburb,Condo,,2642,8452,2,2.5,1986,21.96,6.6,39.5,6.27,8.5,644423.0,667625.0,677707.0
4,500005,Suburb,Condo,Good,1182,13430,1,3.5,2011,0.30,6.8,36.6,5.96,6.5,466989.0,463497.0,473091.0


,ListingID,Neighborhood,PropertyType,Condition,SqFt,LotSqFt,Beds,Baths,YearBuilt,DistanceToCityMiles,SchoolRating,CrimeIndex,MortgageRatePct,TotalRooms,OnlineEstimateUSD,SalePriceUSD,PostSaleAppraisalUSD
895,500896,University,Condo,Good,1279,4344,5,1.0,1962,4.22,8.5,32.8,5.76,8.0,319714.0,361583.0,360041.0
896,500897,Rural,House,Fair,1092,2552,3,3.0,1976,12.05,7.3,54.8,5.80,8.0,237791.0,257075.0,230748.0
897,500898,Suburb,Townhome,Fair,1900,2229,4,2.0,1976,6.37,6.8,60.3,6.17,9.0,512403.0,543375.0,524858.0
898,500899,Rural,Condo,Good,1916,8898,5,1.0,2020,9.27,7.7,58.6,5.44,10.0,480664.0,500486.0,485861.0
899,500900,Downtown,House,Good,1295,3994,2,2.5,2009,11.15,8.1,16.6,5.93,9.5,553490.0,589057.0,592484.0


# 2. Data Quality Checks and Cleaning

## Step 1: Code

In [42]:

# 1) Data Acquisition & "Blank String" Handling
# Load data and immediately replace blank strings with NaN 
# as required by the business scenario
df = pd.read_csv("IST660_IA4_synthetic_real_estate.csv", keep_default_na=False)
df = df.replace("", np.nan)


# 2) Missing Values Summary (Counts + %)
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct.round(2)
})

print("--- Missing Values Summary (All Columns) ---")
display(missing_summary[missing_summary["missing_count"] > 0])


# 3) Duplicates & ID Uniqueness Check
print(f"\nDuplicate FULL rows: {df.duplicated().sum()}")

# Specific Uniqueness check for the ListingID column
print(f"\nUniqueness check for 'ListingID':")
print(f"  Unique values: {df['ListingID'].nunique(dropna=False)}")
print(f"  Duplicated IDs: {df['ListingID'].duplicated().sum()}")

# 4) Uniqueness / Cardinality for each column
# This provides evidence for which columns are IDs, Categories, or Numbers
uniq = df.nunique(dropna=False).sort_values(ascending=False)
uniq_summary = pd.DataFrame({
    "n_unique_including_NA": uniq,
    "pct_unique_including_NA": (uniq / len(df) * 100).round(2)
})
print("\nUniqueness (cardinality) summary for Real Estate Data:")
display(uniq_summary)


# 4) Type Conversion: Numeric Cleaning
# These columns were loaded as 'object' due to the blank strings
cols_to_fix = ['SqFt', 'LotSqFt', 'Baths']

print("\n--- Dtypes Before Cleaning ---")
print(df[cols_to_fix].dtypes)

# Convert to numeric
for col in cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("\n--- Dtypes After Cleaning ---")
print(df[cols_to_fix].dtypes)


--- Missing Values Summary (All Columns) ---


,missing_count,missing_pct
LotSqFt,50,5.56
SqFt,34,3.78
Baths,28,3.11
Neighborhood,14,1.56
Condition,13,1.44



Duplicate FULL rows: 0

Uniqueness check for 'ListingID':
  Unique values: 900
  Duplicated IDs: 0

Uniqueness (cardinality) summary for Real Estate Data:


,n_unique_including_NA,pct_unique_including_NA
ListingID,900,100.00
PostSaleAppraisalUSD,900,100.00
OnlineEstimateUSD,899,99.89
SalePriceUSD,896,99.56
LotSqFt,828,92.00
SqFt,720,80.00
DistanceToCityMiles,674,74.89
CrimeIndex,450,50.00
MortgageRatePct,257,28.56
YearBuilt,76,8.44



--- Dtypes Before Cleaning ---
SqFt       object
LotSqFt    object
Baths      object
dtype: object

--- Dtypes After Cleaning ---
SqFt       float64
LotSqFt    float64
Baths      float64
dtype: object


In [49]:
# ============================================================
# Value Counts for Low-Cardinality Columns
LOW_CARD_THRESHOLD = 30  

low_card_cols = [
    c for c in df.columns 
    if df[c].nunique(dropna=False) <= LOW_CARD_THRESHOLD
]

print(f"\nLow-cardinality columns (<= {LOW_CARD_THRESHOLD} unique):", low_card_cols)

for c in low_card_cols:
    print(f"\nValue counts for '{c}':")
    # This line ensures the table is readable and doesn't cut off
    counts_df = df[c].value_counts(dropna=False).head(20).reset_index()
    counts_df.columns = [c, "count"]
    display(counts_df)


Low-cardinality columns (<= 30 unique): ['Neighborhood', 'PropertyType', 'Condition', 'Beds', 'Baths', 'TotalRooms']

Value counts for 'Neighborhood':


,Neighborhood,count
0,Suburb,369
1,Rural,172
2,Downtown,163
3,University,118
4,Waterfront,64
5,NaN,14



Value counts for 'PropertyType':


,PropertyType,count
0,House,559
1,Condo,221
2,Townhome,120



Value counts for 'Condition':


...

,Beds,count
0,3,333
1,4,268
2,2,180
3,5,71
4,1,39
5,6,9



Value counts for 'Baths':


,Baths,count
0,2.0,212
1,2.5,193
2,1.5,149
3,3.0,134
4,1.0,101
5,3.5,64
6,NaN,28
7,4.0,16
8,5.0,2
9,4.5,1



Value counts for 'TotalRooms':


,TotalRooms,count
0,10.0,105
1,8.5,100
2,9.5,96
3,9.0,93
4,8.0,82
5,7.0,75
6,7.5,74
7,10.5,65
8,11.0,51
9,6.0,34


## Step 2: Interpretation and Justification

### Explanation of “Blank String Missingness" 
In the IST660_IA4_synthetic_real_estate.csv file, missing data was represented by the character sequence "" (an empty string). In Python/Pandas, an empty string is technically a valid string object with zero length. This is problematic for two specific reasons:
1) Metric Failure: Standard diagnostic tools like .isnull().sum() return 0 for these columns, hiding the missingness.
2) Mathematical Block: A column containing even one "" is forced into an object (string) data type, which prevents the calculation of the mean, median, or regression coefficients.

#### Specific Column Handling
Upon inspection, I identified and handled the following columns containing blank strings by standardizing them to NaN:
1) Physical Dimensions: LotSqFt (50 blanks / 5.56%) and SqFt (34 blanks / 3.78%). These are the primary drivers of property value in our regression model.
2) Property Features: Baths (28 blanks / 3.11%), Neighborhood (14 blanks), and Condition (13 blanks).

#### Justification for Type Cleaning (Object to Float64)
The columns SqFt, LotSqFt, and Baths were initially identified as object types because the empty strings forced the entire column to be treated as text. I have converted these to float64 for the following reasons:
1) SqFt & LotSqFt: These must be numeric to calculate coefficients (e.g., the dollar-value increase for every additional square foot).
2) Baths: This column contains fractional values (e.g., 2.5 bathrooms). Using float64 ensures these are treated as continuous variables and prevents the data loss that would occur if they were converted to integers.
##### Compatibility: Converting to float64 allows the columns to hold NaN values, which is a prerequisite for the scaling and imputation steps in the modeling pipeline.

#### Verification of Data Integrity
The check for duplicates confirmed 0 duplicate rows and 900 unique ListingID entries. This proves that the dataset maintains the "one row per unique property listing" definition, ensuring the model's performance isn't biased by redundant observations.

"The cardinality check provides the technical justification for our feature engineering. ListingID shows 900 unique values (100% cardinality), confirming it acts as a unique row identifier and must be dropped to prevent the model from memorizing indices. Conversely, columns like Neighborhood and Condition show very low unique counts (under 2%), confirming they are categorical variables that require one-hot encoding rather than being treated as continuous numbers."

# 3. Train/Test Split and Validation Mindset

## Step 1: Code
This code separates the target from the predictors, removes the non-predictive and leakage columns, and performs the split.

In [52]:

# 1) Define Features (X) and Target (y)
# Drop the Target variable for X
# Drop ListingID (Non-predictive ID)
# Drop PostSaleAppraisalUSD (Identified Data Leakage)
X = df.drop(columns=['SalePriceUSD', 'ListingID', 'PostSaleAppraisalUSD'])
y = df['SalePriceUSD']


# 2) Train/Test Split
# We use a 75/25 split to ensure enough data for training 
# while maintaining a robust set for validation.
# random_state=42 ensures the split is reproducible.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


# 3) Verification of Split
print(f"Total observations: {len(df)}")
print(f"Training set size: {X_train.shape[0]} rows")
print(f"Testing set size:  {X_test.shape[0]} rows")
print(f"Features in X: {X.columns.tolist()}")

Total observations: 900
Training set size: 675 rows
Testing set size:  225 rows
Features in X: ['Neighborhood', 'PropertyType', 'Condition', 'SqFt', 'LotSqFt', 'Beds', 'Baths', 'YearBuilt', 'DistanceToCityMiles', 'SchoolRating', 'CrimeIndex', 'MortgageRatePct', 'TotalRooms', 'OnlineEstimateUSD']


## Step 2: Interpretation and Justification
#### Why We Split into Train/Test
In real estate analytics, a model is only useful if it can accurately estimate the price of a new listing that was not part of the historical training data. If we trained the model on the entire dataset, we would risk "overfitting"—where the model simply memorizes the specific quirks of these 900 properties rather than learning the general relationships (e.g., how square footage generally affects price). By splitting the data, we keep 25% of the listings hidden from the model. This acts as a "unseen" proxy for future market listings, allowing us to validate if our pricing estimates are truly reliable before they are used by agents in the field.
##### Split Configuration and Reproducibility
- **Split Ratio**: I utilized a 75/25 split (75% training, 25% testing). With a dataset of 900 rows, this provides 675 observations for the model to learn relationships (like the impact of SqFt and Neighborhood) and 225 observations to validate performance.
- **Random State**: I set random_state=42. This is critical for scientific reproducibility; it ensures that every time this notebook is run, the exact same rows end up in the training and testing sets, allowing for consistent results when tuning hyperparameters.
- **Feature Exclusion**: * ListingID: Removed because it is a unique identifier with no predictive value.
     - **PostSaleAppraisalUSD**: Removed because it is only available after a sale occurs. Including it would create a false sense of accuracy (leakage) that would not exist when an agent tries to use this model for a new listing.

# 4. Preprocessing Pipeline (Numeric vs Categorical)

## Step 1 : Code

In [64]:

# 1) Identify which columns are numeric (int/float) vs categorical (text/object)
# This uses the cleaned dtypes from Section 2.
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Numeric features identified: {numeric_cols}")
print(f"Categorical features identified: {categorical_cols}")

# 2) Define preprocessing steps for NUMERIC columns
# We use Median to handle outliers in SqFt/LotSqFt and StandardScaler for regression.
numeric_prep = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 3) Define preprocessing steps for CATEGORICAL columns
# We use 'most_frequent' for missing Neighborhood/Condition data.
categorical_prep = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 4) Combine pipelines into one ColumnTransformer
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_prep, numeric_cols),
        ("cat", categorical_prep, categorical_cols)
    ],
    remainder="drop"
)

# 5) Define the full model pipeline
# This chains the transformer and the estimator (LinearRegression) together.
model = Pipeline(steps=[
    ("prep", preprocess),
    ("lr", LinearRegression())
])

print("\n✅ Preprocessing transformer created and `model` pipeline defined.")

Numeric features identified: ['SqFt', 'LotSqFt', 'Beds', 'Baths', 'YearBuilt', 'DistanceToCityMiles', 'SchoolRating', 'CrimeIndex', 'MortgageRatePct', 'TotalRooms', 'OnlineEstimateUSD']
Categorical features identified: ['Neighborhood', 'PropertyType', 'Condition']

✅ Preprocessing transformer created and `model` pipeline defined.


## Step 2: Interpretation and Justification

### Feature Classification
Based on our data inspection and the business context, the features have been partitioned as follows:
- **Numeric Columns**: SqFt, LotSqFt, Beds, Baths, YearBuilt, DistanceToCityMiles, SchoolRating, CrimeIndex, MortgageRatePct, TotalRooms, and OnlineEstimateUSD.
- **Categorical Columns**: Neighborhood, PropertyType, and Condition.

### Why Scaling is Used
The numeric features in this dataset exist on vastly different scales. For instance, OnlineEstimateUSD has a range in the hundreds of thousands, while MortgageRatePct is a small decimal.
- Comparability: By applying the StandardScaler, we transform all features to have a mean of 0 and a standard deviation of 1. This allows us to compare the coefficients directly; we can see which feature (e.g., SqFt vs. DistanceToCityMiles) truly has the strongest mathematical pull on the house price regardless of its original units.
- Model Stability: Scaling prevents features with large raw magnitudes from dominating the objective function. Without it, the regression model can produce "unstable" coefficients that are highly sensitive to slight variations in the data, potentially leading to poor generalization on the test set.
  
The numeric features in this dataset exist on vastly different scales. For instance, OnlineEstimateUSD reaches values over $700,000, while MortgageRatePct is a small decimal (e.g., 0.06 or 6%). Without scaling, the Multiple Linear Regression model might disproportionately weigh features with larger raw magnitudes, leading to unstable coefficients. By applying the StandardScaler, we transform every feature to have a mean of 0 and a standard deviation of 1. This ensures comparability, allowing us to determine which feature has the greatest impact on SalePriceUSD regardless of its original unit of measure.  

### Why One-Hot Encoding is Required for Categories
Regression is a mathematical equation that requires purely numeric inputs. Features like Neighborhood (e.g., 'Suburb', 'Waterfront') or PropertyType (e.g., 'Condo', 'House') have no inherent mathematical value. One-Hot Encoding creates binary "dummy" variables for each category. For example, it turns a single Neighborhood column into separate Neighborhood_Waterfront and Neighborhood_Suburb columns. This allows the model to calculate a specific price premium or discount associated with each specific property attribute.

### Why Imputation is Needed
Our initial data quality check revealed that several key columns—most notably LotSqFt (50 missing), SqFt (34 missing), and Baths (28 missing)—contain null values (converted from blank strings). Because the LinearRegression algorithm cannot process a row containing a NaN, we must use imputation. We use the median for numeric features to remain robust against outliers in property size and the most frequent (mode) for categorical features like Condition. This allows us to keep all 900 observations in our dataset, maintaining the model's statistical integrity.

In [76]:

# Inspect Transformed Data & Feature Names

# 1) Fit on TRAIN, transform TRAIN and TEST
X_train_t = preprocess.fit_transform(X_train)
X_test_t  = preprocess.transform(X_test)

# 2) Build feature names after preprocessing
feature_names = []
feature_names.extend(numeric_cols)

if categorical_cols:
    ohe = preprocess.named_transformers_["cat"].named_steps["onehot"]
    feature_names.extend(ohe.get_feature_names_out(categorical_cols).tolist())

# 3) Convert transformed matrices into DataFrames for inspection
X_train_t_df = pd.DataFrame(X_train_t, columns=feature_names, index=X_train.index)
X_test_t_df  = pd.DataFrame(X_test_t,  columns=feature_names, index=X_test.index)

print(f"Transformed train shape: {X_train_t_df.shape}")
display(X_train_t_df.head())

# Save to CSV as per sample file instructions
X_train_t_df.to_csv("X_train_transformed.csv", index=False)
X_test_t_df.to_csv("X_test_transformed.csv", index=False)

Transformed train shape: (675, 22)


,SqFt,LotSqFt,Beds,Baths,YearBuilt,DistanceToCityMiles,SchoolRating,CrimeIndex,MortgageRatePct,TotalRooms,...,Neighborhood_Rural,Neighborhood_Suburb,Neighborhood_University,Neighborhood_Waterfront,PropertyType_Condo,PropertyType_House,PropertyType_Townhome,Condition_Excellent,Condition_Fair,Condition_Good
613,-0.343467,-0.409782,-0.209429,0.411972,0.472076,-0.534080,-0.673795,-1.273872,1.155465,-0.269877,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
528,2.020328,-0.350185,-1.197995,1.742505,1.203661,1.271312,-0.673795,0.175226,-0.846572,-0.867146,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
815,-0.478043,-0.392754,-0.209429,-0.918561,0.883592,0.063293,-0.196767,0.357996,-2.009046,0.327392,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
380,-0.018901,0.755196,-0.209429,1.742505,-1.585507,1.322515,0.620994,0.260083,1.010156,0.327392,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
300,0.307249,-0.890206,0.779136,-0.253294,1.295109,1.766278,-1.082675,-0.059763,-0.265336,0.327392,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
